# Homework-3 

## Use the Titanic Dataset to Practice Data Wrangling Tasks
### Part 1: Import libaries , load the data, inspect the data, then get the name of the variables and create a small markdown table  with variable names and types (e.g numerical or categorical ordinal, categorical nominal etc.)

### Part 2: 
    - Impute missing values for "Embarked" with the mode (most frequent value)
    - Impute missing value for Age using two following strategies
        - A) Simple mean imputation (fill with the overal mean age)
        - B) Group-wise median imputation: for each combination of Sex and Pclass compute the median age. Impute missing age using the median age of passenger's sex and Pclass group

### Part 3: Create a FamilySize variable (FamilySize = SibSp + Parch +1) and compute:
    - Minimum, Maximum, and mean FamilySize
    - How many passengers were traveling alone
    
### Part 4:  Create a filtered dataframe of all passengers:
    - Who are female
    - Who are male 
    - Who are under 16 years old   
    - Compute the survival rate for the above filtered data
    - Compute survival rate by Sex and Pclass (use groupby and mean) and present the result in a markedown table (rows: sex, columns: Pclass) or using print formating.
    - Interpret your result in 2-3 sentences
     
    
### Part 5: Compute the average fare for each Pclass (Fare values are per tickets not per person (there coud be more than 1 person on a ticket) so make sure to account for that. 

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Variable       Data Type       Classification
PassnegerId    Numerical       Discrete(Identifier)
Survived       Categorical     Nominal(Binary)
Pclass         Categorial      Ordinal(1st, 2nd, 3rd class)
Sex            Categorical     Nominal
Age            Numerical       Continuous
SibSp/Parch    Numerical       Discrete
Fare           Numerical       Continuous
Embarked       Categorical     Nominal


In [8]:
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
print(df[['Age', 'Embarked']].isnull().sum())

Age         0
Embarked    0
dtype: int64


In [9]:
df['FamilySize'] =df['SibSp'] +df['Parch'] + 1

print(f"Min Family Size: {df['FamilySize'].min()}")
print(f"Max Family Size: {df['FamilySize'].max()}")
print(f"Mean Family Size: {df['FamilySize'].mean()}")

alone_count = (df['FamilySize'] == 1).sum()
print(f"Passengers traveling alone: {alone_count}")

Min Family Size: 1
Max Family Size: 11
Mean Family Size: 1.904601571268238
Passengers traveling alone: 537


In [10]:
female_df = df[df['Sex'] == 'female']
male_df = df[df['Sex'] == 'male']
children_df = df[df['Age'] < 16]

print(f"Female Survival Rate: {female_df['Survived'].mean():.2%}")
print(f"Male Survival Rate: {male_df['Survived'].mean():.2%}")
print(f"Child Survival Rate: {children_df['Survived'].mean():.2%}")

survival_pivot = df.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack()
print("\nSurvival Rate by Sex and Class:")
print(survival_pivot)



Female Survival Rate: 74.20%
Male Survival Rate: 18.89%
Child Survival Rate: 59.04%

Survival Rate by Sex and Class:
Pclass         1         2         3
Sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


In [11]:
print("\nInterpretation:")
print("The data reveals a significant survival advantage for female passengers across all classes, "
      "consistent with the 'women and children first' protocol. Additionally, there is a clear "
      "socio-economic trend: passengers in 1st Class had the highest survival rates, while those "
      "in 3rd Class, particularly males, had the lowest chance of survival.")


Interpretation:
The data reveals a significant survival advantage for female passengers across all classes, consistent with the 'women and children first' protocol. Additionally, there is a clear socio-economic trend: passengers in 1st Class had the highest survival rates, while those in 3rd Class, particularly males, had the lowest chance of survival.


In [12]:
df['TicketCount'] = df.groupby('Ticket')['Ticket'].transform('count')
df['IndividualFare'] = df['Fare'] / df['TicketCount']
final_fare_analysis = df.groupby('Pclass')['IndividualFare'].mean()
print(final_fare_analysis)

Pclass
1    43.650347
2    13.322599
3     8.085857
Name: IndividualFare, dtype: float64
